# grads-dict-accumulate-parents — ex1: accumulate per-parent contributions in the reverse-pass grads dict

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `grads-dict-accumulate-parents`. Running the final beacon cell reports progress against the `Backprop: grads dict accumulate parents` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: grads dict accumulate parents` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`grads-dict-accumulate-parents`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "grads-dict-accumulate-parents"
DD_SUBTOPIC = "Backprop: grads dict accumulate parents"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `grads` dict — accumulate into parents — quick refresher

The reverse pass keeps a single `dict[MiniTensor, torch.Tensor]` mapping each node to its accumulated gradient. When the dispatcher computes a new contribution for a parent, it has to ADD to (not overwrite) the existing entry — a parent may receive contributions from multiple children:

```python
grads = {end_node: end_grad}
for node in topo_order_reversed:
    out_grad = grads[node]
    for argnum, parent in node.recipe.parents.items():
        back_fn = BACK_FUNCS.get(node.recipe.func, argnum)
        contribution = back_fn(out_grad, node.array, *node.recipe.args)
        grads[parent] = grads.get(parent, 0) + contribution   # ← THE accumulation
```

**Why `.get(parent, 0) + contribution`.** A parent on first visit isn't in `grads` yet — `get(parent, 0)` seeds with the additive identity. On subsequent visits the existing accumulator is the running sum. `+` is non-mutating, returning a fresh tensor.

**Common bug.** Writing `grads[parent] = contribution` (overwrite) — looks fine on linear graphs but silently drops contributions when a node has two children. `y = x + x` makes `x` a parent of `y` at argnum 0 AND argnum 1; both contributions must be summed.

### Exercise 1 — accumulate per-parent contributions in the reverse-pass grads dict

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the `grads[parent] = grads.get(parent, 0) + contribution` pattern across a parents-list, correctly summing contributions when the same parent appears more than once.
> Keywords: grads-dict, accumulate, parents, get-default, reverse-pass
> ```

**KCs targeted:** `grads-dict-accumulate-parents`, `grad-accumulate-on-leaf`

Implement `accumulate_into_grads(grads, contributions)`. Mutates the `grads` dict in place:

- `grads`: `dict[MiniTensor, torch.Tensor]` — the reverse-pass accumulator. May start empty or may already have entries.
- `contributions`: `list[tuple[MiniTensor, torch.Tensor]]` — a list of `(parent_node, new_gradient)` pairs the dispatcher just computed for one node's outgoing edges.

For each `(parent, g)` in `contributions`:
```
grads[parent] = grads.get(parent, 0) + g
```

Two rules — both critical:

**1. Use `.get(parent, 0)`, not `grads[parent]`.** A parent visited for the first time isn't in the dict yet. `grads[parent]` would raise `KeyError`. `grads.get(parent, 0)` seeds with the additive identity 0, which broadcasts correctly with any-shape tensor.

**2. Use `+`, not `+=` or overwrite.** `+=` mutates the existing tensor (dangerous if the caller holds a reference). Overwriting `grads[parent] = g` drops earlier contributions — exactly the bug that breaks `y = x + x` reverse pass (where `x` is parent twice and both contributions must sum).

Returns `None`. The caller iterates `grads` after this finishes.

**Why this is the load-bearing line of the reverse pass.** Topological-sort the graph, walk it in reverse, and for each node call this function with the dispatcher's per-arg contributions. When the walk reaches a leaf, `grads[leaf]` already holds the fully-summed total derivative — ready to copy into `leaf.grad`.

In [ ]:
def accumulate_into_grads(grads: dict, contributions: list) -> None:
    """Sum each (parent, g) contribution into grads[parent] using get-default-zero."""
    raise NotImplementedError()


def _test_ex1():
    # --- single contribution to an empty grads dict ---
    p1 = MiniTensor(t.zeros(3))
    p2 = MiniTensor(t.zeros(3))
    grads = {}
    g1 = t.tensor([1.0, 2.0, 3.0])
    ret = accumulate_into_grads(grads, [(p1, g1)])
    assert ret is None, 'should return None (mutates in place)'
    assert p1 in grads, 'p1 must be added to grads'
    assert t.allclose(grads[p1], g1), f'grads[p1] = {grads[p1]}'

    # --- two parents, both new ---
    grads = {}
    accumulate_into_grads(grads, [(p1, t.tensor([1.0, 1.0, 1.0])),
                                  (p2, t.tensor([5.0, 5.0, 5.0]))])
    assert len(grads) == 2
    assert t.allclose(grads[p1], t.tensor([1.0, 1.0, 1.0]))
    assert t.allclose(grads[p2], t.tensor([5.0, 5.0, 5.0]))

    # --- THE CRITICAL TEST: same parent appears TWICE → contributions sum ---
    grads = {}
    accumulate_into_grads(grads, [(p1, t.tensor([1.0, 2.0, 3.0])),
                                  (p1, t.tensor([10.0, 20.0, 30.0]))])
    assert t.allclose(grads[p1], t.tensor([11.0, 22.0, 33.0])), (
        f'same-parent contributions must SUM, got {grads[p1]} — '
        'did you overwrite instead of accumulating?'
    )

    # --- contributions accumulate ACROSS calls (parent already in grads) ---
    grads = {p1: t.tensor([100.0, 100.0, 100.0])}
    accumulate_into_grads(grads, [(p1, t.tensor([1.0, 2.0, 3.0]))])
    assert t.allclose(grads[p1], t.tensor([101.0, 102.0, 103.0])), (
        f'pre-existing grads entry must be ADDED to, got {grads[p1]}'
    )

    # --- empty contributions list is a no-op ---
    grads = {p1: t.tensor([7.0, 7.0, 7.0])}
    accumulate_into_grads(grads, [])
    assert t.allclose(grads[p1], t.tensor([7.0, 7.0, 7.0])), 'empty list must not modify grads'

    # --- `+` not `+=`: the original grad tensor must NOT be mutated ---
    original = t.tensor([5.0, 5.0])
    grads = {p1: original}
    original_id = id(original)
    accumulate_into_grads(grads, [(p1, t.tensor([1.0, 1.0]))])
    # After the +, grads[p1] should be a NEW tensor; the original must be untouched.
    assert t.allclose(original, t.tensor([5.0, 5.0])), (
        f'the previous grad tensor must NOT be mutated in place, got {original}; '
        'did you use `+=` instead of `+`?'
    )
    assert id(grads[p1]) != original_id, 'grads[p1] must be re-bound to a new tensor'
    assert t.allclose(grads[p1], t.tensor([6.0, 6.0])), f'sum wrong: {grads[p1]}'

    # --- the y = x + x scenario: parent x appears twice in one node's contributions ---
    # Reverse pass for `y = x + x` produces contributions [(x, grad_out), (x, grad_out)]
    # — argnum 0 and argnum 1 both feed back to x. Both must sum into grads[x].
    x = MiniTensor(t.tensor([1.0, 1.0]), requires_grad=True)
    grad_out = t.tensor([10.0, 10.0])
    grads = {}
    accumulate_into_grads(grads, [(x, grad_out), (x, grad_out)])
    assert t.allclose(grads[x], t.tensor([20.0, 20.0])), (
        f'y = x + x reverse-pass case: grads[x] must equal 2 * grad_out, got {grads[x]}'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def accumulate_into_grads(grads: dict, contributions: list) -> None:
    for parent, g in contributions:
        # .get(parent, 0) seeds first-touch with additive identity;
        # `+` produces a fresh tensor (no in-place mutation of any prior grad).
        grads[parent] = grads.get(parent, 0) + g
```

**`get(parent, 0)` vs `grads[parent]`.** The bare lookup raises `KeyError` on first touch. The `.get(parent, 0)` default seeds the accumulator with the integer 0 — which then broadcasts correctly with the tensor `g` (producing a fresh tensor of the right shape, dtype, and device). Subsequent calls find an actual tensor in the dict and add to it.

**`+` vs `+=`.** `tensor_a + tensor_b` allocates a new tensor; neither operand is mutated. `tensor_a += tensor_b` mutates `tensor_a` in place, which is faster but breaks any caller that holds a reference to the old grad tensor. The reverse-pass dispatcher often hands grad tensors around — mutating them in place corrupts other computations.

**Why `y = x + x` is the canonical stress case.** Both arg-0 and arg-1 of the add point back to the same parent `x`. The dispatcher hands the accumulator two contributions for the same parent in a single call. An overwriting implementation would keep only the last contribution — `dL/dx` would be off by a factor of 2. The test asserts the sum explicitly.

**Composes with leaf-accumulate.** At the end of the reverse walk, the dispatcher copies `grads[leaf]` into `leaf.grad` via the `accumulate_grad` helper from the previous atom. The `grads` dict is the transient working memory; `leaf.grad` is the persistent output.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()